## Notebook

In questo notebook sperimentale analizziamo le principali attività da svolgere per l'homework

### Caricamento delle librerie

Carichiamo le librerie che useremo nel notebook, il modulo **tes** è stato scritto da noi e contiene:
 - una funzione per il calcolo della **DFT** usando *Python*
 - una funzione per il calcolo della **DFT** usando *C*
 - una funzione per effettuare lo **shift** della DFT
 - una funzione per calcolare lo **spettro di energia**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tes, soundfile, os
import time
from IPython.display import clear_output

# importare importlib consente, in caso di modifiche di un modulo di poterlo
# ricaricare tramite: importlib.reload(nome_modulo)
# import importlib

### Lettura del file audio

Leggiamo il file audio per estrarne il campionamento

In [ ]:
file_audio = str(os.path.join('input', 'Cornfield Chase.wav'))

audio, fs = soundfile.read(file_audio)
audio = np.array(audio[:, 0], dtype=np.float32)

print(f"Estratti {len(audio)} campioni - campionati a {fs/1000:.1f}kHz - durata {len(audio)/fs:.2f} secondi")

### Definizione dei parametri

Definiamo qui i parametri su cui lavoreremo, fra questi abbiamo il numero di campioni che andremo ad analizzare e la conseguente risoluzione in frequenza; calcoliamo ora anche la scala dell'asse X che plotteremo in seguito.

In [ ]:
N = int(fs/2)   #finestra da 0.5 secondi

Df = 1/(N*1/fs)
x_axis = [x*Df for x in range(int(-N/2),int(N/2))]

print(f"Lavoriamo su {N} campioni - risoluzione in frequenza {Df:.2f}Hz")

### Calcolo della FFT tramite funzioni di libreria

Adoperiamo qui la funzione *fft* del modulo **NumPy** inclusa in Python per calcolare la DFT in maniera efficiente e plottarne lo spettro.

In [ ]:
for i in range(0,int(len(audio)/N)):
    clear_output(wait=True)   
    # calcoliamo la FFT sugli N campioni compresi tra 1 e 2 secondi (24000 campioni su 48000 campioni al secondo)
    fft_py = np.abs(np.fft.fft(audio[i*N:i*N+N]))

    # calcoliamo lo spettro
    spettro_ftt = tes.get_spettro(fft_py)

    # calcoliamo il limite di frequenza per avere la banda al 99% del segnale
    max_freq_banda = tes.get_limite_banda(spettro_ftt, Df, 99)
    print(f"La frequenza corrispondente alla banda al 99% è {max_freq_banda}Hz")

    # plottiamo lo spettro del segnale
    plt.plot(x_axis, np.fft.fftshift(fft_py))
    plt.grid(True)
    plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(fft_py)])
    plt.xlabel("Frequency [Hz]")
    plt.ylabel("Amplitude")
    plt.show()
    time.sleep(0.5)

### Implementazione della DFT

L'implementazione della DFT, presente nel modulo **tes** presenta due versioni. La prima, scritta interamente in Python, viene qui chiamata e presenta come logica algoritmica quella vista a lezione per cui si procede a sommare per ogni frequenza le rispettive componenti lungo tutto il segnale.

Sempre in Python è stata anche implementata una funzione di **shift** della DFT, utile a ricomporre il risultato della trasformata per ridarle il significato reale.

Malgrado la semplicità offerta da Python ci si scontra contro i limiti pratici del linguaggio: di fatti il calcolo anche per numeri di campioni non eccessivamente elevati (nell'ordine di migliaia) può richiedere anche decine di secondi se non minuti.

In [ ]:
# calcoliamo la FFT sugli N campioni compresi tra 1 e 2 secondi (24000 campioni su 48000 campioni al secondo)
dft_py = np.abs(tes.dft_python(audio[N:2*N]))

# calcoliamo lo spettro
spettro_dft_py = tes.get_spettro(dft_py)

# calcoliamo il limite di frequenza per avere la banda al 99% del segnale
max_freq_banda = tes.get_limite_banda(spettro_dft_py, Df, 99)
print(f"La frequenza corrispondente alla banda al 99% è {max_freq_banda}Hz")

# plottiamo lo spettro del segnale
plt.plot(x_axis, tes.shift(dft_py))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(dft_py)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.show()

### Implementazione in C

Spinti dall'inefficienza della soluzione precedente ci siamo chiesti quanto si possa imputare a Python e quanto invece, come esposto a lezione, fosse da imputare all'inefficienza dell'algoritmo.

Abbiamo pertanto, usando **ctypes**, implementato un wrapper in Python che richiama una funzione scritta in C che svolgesse le stesse operazioni della versione in Python.

I risultati sono **decisamente** migliori se confrontati con la funzione in Python anche se, ovviamente, non sono minimamente paragonabili alla FFT che compie i calcoli in una frazione infinitesimale del tempo richiesto dalla DFT.

Pertanto sì, anche se una gran parte dell'inefficienza del calcolo è da appuntare a Python, si può dire che la FFT presenta un vantaggio **notevole** nei calcoli.

In [ ]:
# calcoliamo la FFT sugli N campioni compresi tra 1 e 2 secondi (24000 campioni su 48000 campioni al secondo)
dft_c = tes.dft_c(audio[N:2*N]) # qui non chiamiamo abs perché viene già calcolato da C

# calcoliamo lo spettro
spettro_dft_c = tes.get_spettro(dft_c)

# calcoliamo il limite di frequenza per avere la banda al 99% del segnale
max_freq_banda = tes.get_limite_banda(spettro_dft_c, Df, 99)
print(f"La frequenza corrispondente alla banda al 99% è {max_freq_banda}Hz")

# plottiamo lo spettro del segnale
plt.plot(x_axis, tes.shift(dft_c))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(dft_c)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.show()

### E Matlab?

Per quanto Matlab sia un linguaggio di programmazione altamente efficientato per il calcolo numerico ci si scontra sempre contro una realtà di fondo: è un linguaggio interpretato perciò anche qui la DFT implementata a mano non risulta particolarmente più efficiente della versione nativa in Python.

Si riporta in seguito il codice

In [ ]:
function fourier = dft(segnale)
  % creiamo il segnale
  dimSegnale = length(segnale);
  fourier = zeros(dimSegnale);

  % facciamo la dft
  for k = 1:dimSegnale
    somma = 0;

    for n = 1:dimSegnale
      somma = somma + segnale(n) * exp(-1i * 2 * pi * n * k / dimSegnale);
    endfor

    fourier(k) = somma;
  endfor
end

## Spettro di energia
Procediamo ora a calcolare lo spettro di energia del segnale in analisi. Lo spettro di energia rappresenta il contenuto di energia associato a ogni frequenza del segnale.
Si noti che nei passaggi precedenti abbiamo già fatto uso del concetto di spettro di energia per definire la banda al 99%, adesso andremo ad esplicitarne il calcolo. 
Per il calcolo dello spettro di energia partiremo dal risultato della fft, ricordando comunque che si poteva usare uno qualsiasi degli algoritmi implementati fin ora, in quanto sono perfettamente equivalenti dal punto di vista dell'output generato.

In [ ]:
#Implementiamo il calcolo dello spettro di energia
spettro = np.square(fft_py)

plt.plot(x_axis, tes.shift(spettro))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(spettro)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Energy")
plt.show()

Proviamo anche a sperimentare con una scala logaritmica sull'asse y.  
Qui possiamo vedere come anche le frequenze che nel grafico lineare sembrano avere zero energia in verità hanno un piccolo valore di energia.  
Questo accade poichè il segnale che stiamo usando è (ovviamente) un segnale reale, e quindi limitato nel tempo.  
Un segnale limitato nel tempo però risulta essere sempre illimitato in frequenza e di conseguenza ogni frequenza avrà un certo quantitativo (magari irrisorio) di energia

In [ ]:
plt.plot(x_axis, tes.shift(spettro))
plt.grid(True)
plt.yscale('log')
plt.xlabel("Frequency [Hz]")
plt.ylabel("Energy")
#plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(spettro)])
plt.show()